In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import statsmodels.api as sm
import shapely
from tqdm.contrib.concurrent import process_map
from glob import glob
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 255)
pd.set_option('display.max_rows', 100)

In [3]:
files = glob(r"Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\*.shp", recursive=True)
files = pd.Series(files)
#files = files[~files.str.contains("SANNE_MANGROVES") & ~files.str.contains("Join_shapefiles")].reset_index(drop=True)
files

0    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_02FEB2020.shp
1    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_03FEB2006.shp
2    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_18JAN2014.shp
3    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20DEC2011.shp
4    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20JAN2017.shp
5    Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_26JUN2024.shp
dtype: str

In [4]:
CPS_error_lookup = {1: 0.43, 2: 0.73, 3: 0.97, 4: 2.07, 5: 8.59}
def calc_UNCY(row):
  # Calculate Total_UNCY
  Ep = row.Pixel_Er
  Ed = CPS_error_lookup[row.CPS]
  Eg = row.Georef_ER
  return np.sqrt(Ep**2 + Eg**2 + Ed**2)

for f in tqdm(files):
  try:
    df = gpd.read_file(f)
  except Exception as e:
    print(f"Error reading {f}: {e}")
    continue
  if df.empty:
    continue
  try:
    df['Total_UNCY'] = df.apply(calc_UNCY, axis=1)
  except Exception as e:
    print(f"Error calculating Total_UNCY for {f}: {e}")
    continue
  
  try:
      df.to_file(f)
  except Exception as e:
      print(f"Can't write {f} - {e}")

 83%|████████▎ | 5/6 [00:02<00:00,  2.30it/s]c:\Users\erya008\envs\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
100%|██████████| 6/6 [00:02<00:00,  2.13it/s]
